In [2]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to DuckDB
con = duckdb.connect("../data/olist.duckdb", read_only=True)     # read-only to prevent accidental writes

# Helper to run a query and get a DataFrame
def q(sql):
    return con.execute(sql).fetchdf()

# 1. Analysis
This notebook will be answering the five business questions defined in `docs/data_model.md`, using the star schema built in `02_cleaning_model.ipynb`. Findings from this notebook will feed into a Power BI dashboard and the stakeholder report.

**Main question**: *How does delivery performance vary across regions, sellers, and product categories, and how does it impact customer satisfaction?*

## Q1. Olist's order fulfiment rate
**Question**: *"What is Olist's order fulfilment rate?"*

**Why it matters**: Knowing Olist's order fulfilment rate shows a company's overall performance and will help identify problem areas where improvements can be made.

**Method**: To answer this question, we will only be using `fact_deliveries` by calculating the percentage of on-time deliveries to the total number of deliveries made. These are the conditions that qualifies a delivery as "fulfiled":
- `order_status` is `delivered`
- `estimated_delivery_date_key` is not empty
- `is_late` is False

### Queries

In [3]:
# Get count of total deliveries
total = con.execute("""
    SELECT COUNT(*) AS total_deliveries
    FROM mart.fact_deliveries
    WHERE is_delivery_complete = TRUE
""").fetchone()[0]                                              

# Get count of on-time deliveries
on_time = con.execute("""
    SELECT COUNT (*) AS on_time
    FROM mart.fact_deliveries
    WHERE is_delivery_complete = TRUE
                      AND NOT is_late     
""").fetchone()[0]

# Calculate order fulfilment rate
fulfilment_rate = on_time / total * 100

print(f"On-time deliveries:  {on_time:,}")
print(f"Total completed:     {total:,}")
print(f"Fulfilment rate:     {fulfilment_rate:.2f}%")

On-time deliveries:  101,460
Total completed:     110,173
Fulfilment rate:     92.09%


Based on this result we can see that Olist had a total of 110,173 successful deliveries and 101,460 on-time deliveries. This gives us a 92.09% of order fulfilment rate. This means that there were 8,713 deliveries that were delivered late. We dig deeper into this number below:

In [4]:
# Check late deliveries
late_breakdown = con.execute("""
    SELECT 
                             dd.year AS delivery_year,
                             COUNT (*) AS late_deliveries
    FROM mart.fact_deliveries f
    JOIN mart.dim_date dd ON f.delivery_date_key = dd.date_key
    WHERE is_late = TRUE
    GROUP BY dd.year
    ORDER BY dd.year
""").fetchdf()

late_breakdown

,delivery_year,late_deliveries
0,2016,6
1,2017,2419
2,2018,6290


Based on this result, we can see that there 6 late deliveries in 2016. In 2017 there were 2,419 late deliveries, and 6,290 in 2018. Each year the number goes up significantly than the last. A 2,400+ increase of late deliveries from 2016 to 2017, and almost 4,000 increment of late deliveries from 2017 to 2018. 

However, we also need to see the **late rate per year** instead of just the raw count.

In [5]:
late_rate_by_year = con.execute("""
    SELECT 
        dd.year,
        COUNT(*) AS total_completed,
        COUNT(*) FILTER (WHERE is_late) AS late_deliveries,
        100.0 * COUNT(*) FILTER (WHERE is_late) / COUNT(*) AS late_rate_pct
    FROM mart.fact_deliveries f
    JOIN mart.dim_date dd ON f.delivery_date_key = dd.date_key
    WHERE is_delivery_complete = TRUE
    GROUP BY dd.year
    ORDER BY dd.year
""").fetchdf()

late_rate_by_year

,year,total_completed,late_deliveries,late_rate_pct
0,2016,317,6,1.892744
1,2017,46787,2418,5.168102
2,2018,63069,6289,9.971618


Based on this result we can see that Olist's logistics performance really was degrading. The number of deliveries in 2016 was significantly lower than 2017 and 2018, due to the data starting in September. The real performance to look at is between 2017 and 2018 where the rate of late deliveries almost doubled from 5% to 9.9%. This considerable increase in rate is worth further investigation to check whether there was a problem or not.

### Analysis
Overall, Olist has a good order fulfilment rate of 92.09%. Out of 110,173 orders sucessfuly delivered, 101,460 were delivered on time. That is 8,713 of the deliveries were late. 92% of order fulfilment rate is not a bad number. However Olist should aim to be as close to 100% as possible. 

Digging deeper into these deliveries we can see that there were 6 late deliveries in 2016, 2,419 in 2017, and 6,290 in 2018. The late deliveries increased each year significantly by the thousands. However, 2016's data incomplete, so it's not a reliable source of information.

Looking further into the rate of late deliveries we can see that between 2017 and 2018, the rate of late deliveries almost doubled from 5% to 9.9%. This considerable increase is worth looking deeper into to figure out the cause.

In [6]:
con.close()